# **Categorización intereses de los usuarios**

Este notebook utiliza un LLM para identificar los principales intereses temáticos de los usuarios clasificados previamente como Positivo en la evaluación de riesgo suicida.

Cada usuario es analizado a partir del conjunto completo de sus publicaciones y se le asignan una o varias categorías oficiales de interés, junto con la evidencia textual que justifica cada asignación.

## **Obtención DataFrame**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

tweets_user_id = pd.read_csv(
    '/content/drive/MyDrive/TFM/7. Riesgo_suicidio/input_categorizacion.csv'
)

tweets_user_id.head()

,user_id,tweets_concat,n_tweets
0,01fc52c4bc9ffc8c,[TWEET 1] Cusndo sera el día que yo diga “no m...,10
1,04d09936041bc786,"[TWEET 1] Mi padre tiene un tumor, le operan e...",2
2,05306b7fa750a5d8,[TWEET 1] Estoy muy resfriado y tengo chuchos ...,1
3,0532b08c30376fde,[TWEET 1] llevo todo el día en la cama\n\n[TWE...,32
4,0588fd52a2609179,[TWEET 1] RT [USUARIO]: Cuando tu propia famil...,4


In [ ]:
tweets_psico = pd.read_csv(
    '/content/drive/MyDrive/TFM/8. Categorizacion/clasificacion_psicologos.csv'
)

tweets_psico.head()

,tweets_concat,clasif_modelo,clasif_experto
0,[TWEET 1] DEP...,Dudoso,Dudoso
1,[TWEET 1] RT [USUARIO]: Ya sé que no me lee na...,Dudoso,Dudoso
2,[TWEET 1] Stoy faking triste y a punto de puto...,Positivo,Dudoso
3,[TWEET 1] Cusndo sera el día que yo diga “no m...,Positivo,Positivo
4,[TWEET 1] Alguien me espía\n\n[TWEET 2] RT [US...,Dudoso,Dudoso


In [ ]:
tweets_psico = pd.merge(
    tweets_psico,
    tweets_user_id[['tweets_concat', 'user_id']],
    on='tweets_concat',
    how='inner'
)

tweets_psico.shape

(73, 4)

In [ ]:
tweets_agrupados = pd.read_csv(
    '/content/drive/MyDrive/TFM/6. Agrupacion_usuarios/agrupacion_usuarios.csv'
)

tweets_agrupados.head()

,user_id,userName,name,tweets_concat,n_tweets,first_tweet,last_tweet,profile_bio,location,followers,following,lang
0,003aa0c7bf89f9e3,USR_9b3b8ec2,[NOMBRE],[TWEET 1] RT [USUARIO]: Los inmigrantes regula...,93,2026-04-02,2026-04-18,Madre de familia y activista para la protecció...,<LOCATION>,337.0,1554.0,es
1,009c8bb24862d606,USR_0bb8dab2,[NOMBRE],"[TWEET 1] RT [USUARIO]: Esto es mi BCN, noche ...",99,2017-08-17,2026-03-27,NaN,Currently in <LOCATION> 🇪🇸,172.0,531.0,es
2,00bcbd0b067ef737,USR_0204aec3,[NOMBRE],"[TWEET 1] RT [USUARIO]: ‼️Alto y claro, <PERSO...",98,2026-04-19,2026-04-20,"Viajar, conocer lugares y su gente es mi pasión.","<LOCATION>, <LOCATION>",706.0,672.0,es
3,00fa95048919fd68,USR_8405c455,[NOMBRE],[TWEET 1] i would love to check my phone here\...,67,2024-10-06,2026-04-19,🔗🔗🔗🔗,en la silla de la tarta :$,125.0,458.0,en
4,011a3468f9d54417,USR_d78d028e,[NOMBRE],[TWEET 1] Mala Gente.....!!!!! [URL]\n\n[TWEET...,21,2023-01-03,2023-01-12,NaN,<PERSON>,1098.0,1128.0,es


In [ ]:
cols_to_drop = [col for col in tweets_agrupados.columns if 'clasif_modelo' in col or 'clasif_experto' in col]
tweets_agrupados = tweets_agrupados.drop(columns=cols_to_drop, errors='ignore')

tweets_agrupados = pd.merge(
    tweets_agrupados,
    tweets_psico[['user_id', 'clasif_modelo', 'clasif_experto']],
    on='user_id',
    how='inner'
)

tweets_agrupados.head()

,user_id,userName,name,tweets_concat,n_tweets,first_tweet,last_tweet,profile_bio,location,followers,following,lang,clasif_modelo,clasif_experto
0,01fc52c4bc9ffc8c,USR_69f55ce3,[NOMBRE],[TWEET 1] Nacemos solos &amp; morimos en el al...,74,2025-07-22,2026-04-16,NaN,"<LOCATION>, <LOCATION>",788.0,230.0,es,Positivo,Positivo
1,04d09936041bc786,USR_22126242,[NOMBRE],[TWEET 1] RT [USUARIO]: Si ante la desgracia d...,100,2023-12-08,2026-04-16,"Psicóloga. Especializada en trauma, trastorn...",NaN,9186.0,351.0,es,Dudoso,Positivo
2,05306b7fa750a5d8,USR_e1786264,[NOMBRE],[TWEET 1] Si no dice Israel en el título no es...,100,2025-07-28,2026-04-20,"Orgullosamente judio. Si sos nazi, zurdo o k n...","<LOCATION>, <LOCATION>",8515.0,4776.0,es,Positivo,Positivo
3,0532b08c30376fde,USR_7ff3b9c0,[NOMBRE],[TWEET 1] llevo todo el día en la cama\n\n[TWE...,99,2024-01-21,2024-12-18,ig;:? ñ,"<LOCATION>, <LOCATION>",26.0,140.0,es,Positivo,Positivo
4,0588fd52a2609179,USR_8794a5e3,[NOMBRE],[TWEET 1] RT [USUARIO]: Optimus Christ [URL]\n...,98,2026-04-07,2026-04-20,Androide power!!! Obsesionado de la tecnología...,"<LOCATION>, <LOCATION>",709.0,3111.0,en,Dudoso,Positivo


In [ ]:
tweets_filtrados = tweets_agrupados[tweets_agrupados['clasif_experto']=='Positivo']
tweets_filtrados.shape

(73, 14)

## **Modelo** 

El objetivo es caracterizar el perfil temático de los usuarios mediante una clasificación basada en un catálogo de 198 categorías oficiales.

Esta información permite analizar qué temas aparecen con mayor frecuencia en los perfiles de usuarios con riesgo suicida y facilita posteriores análisis estadísticos y exploratorios.

- Hasta 10 categorías por usuario (puede ser 0 si no hay evidencia real).
- Flag explícito `sin_categoria` cuando ninguna aplica.
- Categorías oficiales preservadas sin modificar (incluyendo trailing spaces y duplicados); el modelo las identifica por ID numérico.
- Cada categoría devuelta lleva evidencia textual para auditoría.

In [ ]:
!pip install -q -U google-generativeai tqdm pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.7/58.7 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 21.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 2.13.4 which is incompatible.


In [ ]:
import json
import time
import os
import ast
import google.generativeai as genai
import pandas as pd
from google.colab import userdata
from pydantic import BaseModel, Field
from typing import List
from tqdm import tqdm

try:
    GOOGLE_API_KEY = userdata.get('LAURA_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)
    print("API Key configurada correctamente.")
except Exception as e:
    print(f"ERROR: Configura 'GOOGLE_API_KEY' en los Secrets de Colab. Detalle: {e}")

API Key configurada correctamente.


In [ ]:
CATEGORIAS_C = [
    'Equipamiento para actividades al aire libre','Equipación deportiva','Comida','Alimentos y productos frescos','Bebidas','Cocina y recetas','Negocios y economia ','Ciencia y tecnología ','Cultura e historia','Entidades beneficas y de bienestar social','Derecho','Protección del medio ambiente ','Famosos','Noticias de estilo de vida','Eventos en directo','Objetos de colección y antiguedades','Telenovelas y series de TV','Anime','Astrologia','Cultura y arte','Programas de television de variedades','Lectura ','Peliculas ','Belleza y cuidado personal ','Deporte y ejercicio ','Comida y cocina ','Automoviles ','Mascotas','Relaciones','Turismo ','Otros de noticias ','Salud y bienestar','Noticias de coches','Hoteles y alojamiento','Tours y atracciones','Agencias de viajes y servicios','Automoviles ','Accesorios para automoviles ','Servicios para automoviles ','Piezas para automoviles ','Alquiler de automoviles ','Automoviles de segunda mano ','Motocicletas','Bicicletas','Escuteres electricos ','Vehiculos acuaticos','Accesorios para vehículos de 2 o 3 ruedas','Concesionarios de automoviles','Cochecitos y cunas ','Fórmula infantil ','Pañales y toallitas para bebe','Ropa de cama de bebe','Zapatos de bebe ','Juguetes infantiles ','Sillas de coche infantiles','Productos para la alimentación de bebés','Ropa infantil','Telefonos moviles','Ordenadores','Componentes informaticos','Videoconsolas','Accesorios para ordenadores','Productos de red ','Equipo de oficina','Juegos(por tipo)','Monetización','Plataforma','Dimensión ','Tema','Juego ','Educación infantil y preescolar','Educación primaria y secundaria','Educación superior','Educación en el extranjero','Formación profesional','Formación de idiomas ','Formación no académica','Comida para mascotas ','Salud para mascotas','Productos domesticos para mascotas ','Aseo de mascotas ','Juguetes para mascotas','Accesorios de viaje para mascotas','Electrodomésticos de cuidado personal','Electrodomésticos de cocina y baño','Electrodomésticos grandes','Electrodomésticos para el hogar ','Dispositivos digitales','Fotografia de bodas','Fotografía','Regalos y flores','Trabajo doméstico','Servicios de compra','Plataforma de venta de artículos de segunda mano','Jardineria','Servicios de internet ','Servicios al consumidor ','Ejercicio y fitness','Belleza y cuidado personal ','Productos de almacenamiento','Vasos y recipientes','Accesorios de Café','Accesorios de cocina','Electrodomesticos de limpieza','Pañuelos y toallitas humedas','Colada','Cuidado del cuero','Articulos de limpieza','Juguetes ','Control de plagas','Accesorios de ropa','Bolsos','Relojes ','Joyeria corriente','Joyeria de gama alta','Ropa para hombre ','Zapatos de hombre ','Ropa para mujer','Zapatos de mujer','Ropa tradicional y de gala ','Dispositivos tecnológicos ponibles','Redes sociales','Reproductores de audio y de video','Noticias y lectura','Fotografia','Educación','Compra en linea','Servicios financieros','Vida y Ocio ','Viajes','Salud y ejercicio','Negocios y productividad ','Crianza de los hijos','Herramientas','Valores','Seguros ','Oficinas de crédito','Transacciones en divisas','Pagos de terceros','Bienes raíces y alquiler de hogares','Equipo y artículos de oficina','Contratación y búsqueda de empleo','Ingenieria de la construcción','Electronica y electricidad ','Maquinaria y equipo ','Franquicias','Agricultura, silvicultura, ganadería y pesca','Conservación energética y protección del medio ambiente','Seguridad y protección','Logistica','Marketing y publicidad ','Consulta profesional','Servicios de exposición','Servicios informaticos','Servicios juridicos','Servicios de subasta','Planificación y gestión de inversiones','Emprendimiento individual','Higiene bucal ','Cuidado del cabello','Peinado del cabello y pelucas','Cuidado de la piel ','Cosmetica','Fragancias y perfumes ','Cuidado femenino','Grandes tiendas','Comida y bebida','Viajes','Vida cotidiana','Relaciones','Familia','Entretenimiento y cultura','Cosplay ','Juegos ','Anime y comics','Deportes','Fitness y salud','Actividades al aire libre','Sociedad','Animales ','Medio ambiente','Canto y baile','Arte','Magia ','Efectos especiales profesionales','Bricolaje y manualidades ','Otros talentos','Contenido comico','Lipsync','Dramas con guión','Belleza y cuidado personal ','Conjuntos de ropa','Otros de moda','Cultura','Educación','Motivación','Tecnología ','Fotografía'
]

CATEGORIAS_NUMERADAS = "\n".join(f"[{i+1}] {cat}" for i, cat in enumerate(CATEGORIAS_C))

class CategoriaSeleccionada(BaseModel):
    id: int = Field(description="ID numérico entre 1 y 198.")
    nombre: str = Field(description="Nombre exacto de la categoría de la lista.")
    evidencia: str = Field(description="Tema del tweet que justifica esta categoría (máx. 8 palabras).")

class ClasificacionUsuario(BaseModel):
    user_id: str = Field(description="user_id tal como viene en el input.")
    categorias: List[CategoriaSeleccionada] = Field(
        description="Hasta 10 categorías con evidencia real. Lista vacía si ninguna aplica."
    )
    sin_categoria: bool = Field(
        description="True si no hay evidencia real para ninguna categoría."
    )

BATCH_SIZE = 1
RUTA_CHECKPOINT = '/content/drive/MyDrive/TFM/8. Categorizacion/llm_clasificacion_psico_checkpoint_prompt2.csv'
RUTA_RESULTADO  = '/content/drive/MyDrive/TFM/8. Categorizacion/llm_clasificacion_psico_final_prompt2.csv'

print(f"Categorías configuradas: {len(CATEGORIAS_C)}")
print(f"Usuarios a clasificar: {len(tweets_filtrados)}")

Categorías configuradas: 198
Usuarios a clasificar: 73


In [ ]:
from google.generativeai.types import HarmCategory, HarmBlockThreshold

model = genai.GenerativeModel(
    model_name='gemini-3.1-flash-lite',
    safety_settings={
        HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
        HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
        HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_NONE,
        HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
    }
)

generation_config = genai.GenerationConfig(
    response_mime_type="application/json",
    response_schema=ClasificacionUsuario,
    temperature=0.1,
)

print("Modelo listo.")

Modelo listo.


In [ ]:
def _clean_field(v):
    """Devuelve string vacío si es NaN/None, str() en otro caso."""
    if v is None:
        return ""
    s = str(v)
    if s.lower() == "nan":
        return ""
    return s.strip()

print("Función auxiliar lista.")

Función auxiliar lista.


In [ ]:
def clasificar_batch_usuarios(batch_df):
    usuarios_input = [
        {
            "user_id": _clean_field(row["user_id"]),
            "tweets":  _clean_field(row["tweets_concat"]),
        }
        for _, row in batch_df.iterrows()
    ]

    prompt = f"""
Eres un especialista en segmentación de audiencias para Twitter/X Ads.
Tu tarea es identificar los intereses estables y recurrentes de un usuario a partir de sus publicaciones.

OBJETIVO:
Debes inferir únicamente aquellos temas que representen intereses reales, hábitos, aficiones, actividades o áreas de consumo del usuario.
La salida debe reflejar categorías que podrían utilizarse razonablemente para segmentar publicidad o contenido personalizado para ese usuario.
No debes inferir intereses basándote en menciones aisladas, ejemplos, bromas o referencias circunstanciales.

CONTEXTO:
- Los tweets están anonimizados.
- Los marcadores [USUARIO], [URL], <PERSON>, <LOCATION> no aportan información y deben ignorarse.
- Analiza el conjunto completo de tweets del usuario.
- Busca patrones temáticos recurrentes.
- Prioriza intereses observados en múltiples publicaciones.

IMPORTANTE:
Un interés representa una temática estable o recurrente del usuario.
Para asignar una categoría debe existir evidencia de que el usuario:
- habla repetidamente sobre el tema
- consume contenido relacionado
- participa en actividades relacionadas
- sigue acontecimientos relacionados
- expresa afición o interés explícito
- comparte experiencias personales relacionadas con esa temática

NO SON EVIDENCIA SUFICIENTE:
- No clasifiques una categoría únicamente porque aparezca una palabra relacionada.
- Las menciones aisladas NO implican interés.

Ejemplos:
"Devuelve el perro porque irás presa"
→ NO implica Mascotas

"Me atropelló un coche"
→ NO implica Automóviles

Las referencias anecdóticas, noticias, ejemplos, metáforas, insultos, bromas o conversaciones puntuales NO constituyen evidencia suficiente.

RETWEETS:
- Los retweets pueden utilizarse como evidencia únicamente si aparecen de forma consistente dentro de una misma temática.
- Un retweet aislado NO debe considerarse suficiente para inferir un interés.

SELECCIÓN DE CATEGORÍAS:
- Devuelve un máximo de 10 categorías.
- Ordénalas de mayor a menor relevancia.
- Es preferible devolver pocas categorías sólidas que muchas categorías débiles.
- Si no existe evidencia suficiente para ninguna categoría, devuelve una lista vacía y marca sin_categoria=true.
- Para categorías similares, selecciona únicamente la más representativa.
- Usa exclusivamente categorías de la taxonomía proporcionada.
- No inventes categorías nuevas.

CRITERIO DE CONFIANZA:
- Una única referencia solo debe utilizarse cuando sea extremadamente explícita.
Ejemplos válidos:
"Juego a FIFA todas las noches"
→ Juegos

"Estoy entrenando para una maratón"
→ Deporte y ejercicio

"Trabajo como desarrollador de software"
→ Ciencia y tecnología

En caso de duda, NO asignes la categoría.

TAXONOMÍA OFICIAL (198 categorías):
{CATEGORIAS_NUMERADAS}

USUARIOS A CLASIFICAR:
{json.dumps(usuarios_input, ensure_ascii=False)}
"""

    response = model.generate_content(prompt, generation_config=generation_config)
    output = json.loads(response.text)

    if isinstance(output, list):
        output = output[0] if output else {}

    return [output] if output else []

print("Función de clasificación lista.")

Función de clasificación lista.


In [ ]:
def normalizar_resultado(resultado_raw):
    user_id = str(resultado_raw.get("user_id", ""))
    categorias_raw = resultado_raw.get("categorias", []) or []

    vistos_ids = set()
    categorias_clean = []
    for cat in categorias_raw:
        try:
            cid = int(cat.get("id"))
        except (TypeError, ValueError):
            continue
        if cid < 1 or cid > len(CATEGORIAS_C):
            continue
        if cid in vistos_ids:
            continue
        vistos_ids.add(cid)
        categorias_clean.append({
            "id": cid,
            "nombre_oficial": CATEGORIAS_C[cid - 1],
            "nombre_modelo":  str(cat.get("nombre", "")),
            "evidencia":      str(cat.get("evidencia", "")),
        })
        if len(categorias_clean) == 10:
            break

    return {
        "user_id":       user_id,
        "n_categorias":  len(categorias_clean),
        "sin_categoria": len(categorias_clean) == 0,
        "categorias":    categorias_clean,
    }

print("Validador listo.")

Validador listo.


In [ ]:
# CARGAR CHECKPOINT SI EXISTE
if os.path.exists(RUTA_CHECKPOINT):
    df_ckpt = pd.read_csv(RUTA_CHECKPOINT, dtype={"user_id": str})
    ids_ya_procesados = set(df_ckpt["user_id"].tolist())
    resultados_totales = df_ckpt.to_dict(orient="records")
    print(f"✓ Checkpoint: {len(ids_ya_procesados)} usuarios ya procesados. Reanudando...")
else:
    ids_ya_procesados = set()
    resultados_totales = []
    print("Sin checkpoint. Iniciando desde cero.")

df_pendiente = tweets_filtrados[
    ~tweets_filtrados["user_id"].isin(ids_ya_procesados)
].reset_index(drop=True)

print(f"Pendientes: {len(df_pendiente)} | Procesados: {len(ids_ya_procesados)} | Total: {len(tweets_filtrados)}")

# LOOP
for i in tqdm(range(0, len(df_pendiente), BATCH_SIZE)):
    batch = df_pendiente.iloc[i:i + BATCH_SIZE]
    user_id_actual = batch["user_id"].iloc[0]

    exito = False
    for intento in range(3):
        try:
            resultados_batch = clasificar_batch_usuarios(batch)
            for r in resultados_batch:
                normalizado = normalizar_resultado(r)
                resultados_totales.append({
                    "user_id":      normalizado["user_id"],
                    "n_categorias": normalizado["n_categorias"],
                    "sin_categoria": normalizado["sin_categoria"],
                    "categorias":   json.dumps(normalizado["categorias"], ensure_ascii=False),
                })
                if normalizado["sin_categoria"]:
                    print(f"\n[INFO] user_id {normalizado['user_id']}: sin evidencia clara.")
                elif normalizado["n_categorias"] < 10:
                    print(f"\n[INFO] user_id {normalizado['user_id']}: {normalizado['n_categorias']} categorías.")
            exito = True
            break

        except Exception as e:
            espera = 10 * (2 ** intento)
            print(f"\n[Error usuario {user_id_actual}, intento {intento+1}/3]: {e}. Esperando {espera}s...")
            time.sleep(espera)

    if not exito:
        print(f"\n[CRÍTICO] Usuario {user_id_actual} saltado tras 3 intentos.")

    if resultados_totales:
        pd.DataFrame(resultados_totales).to_csv(RUTA_CHECKPOINT, index=False)

    time.sleep(1.5)

print(f"\n✓ Clasificación completada. Total resultados: {len(resultados_totales)}")

Sin checkpoint. Iniciando desde cero.
Pendientes: 73 | Procesados: 0 | Total: 73


  0%|          | 0/73 [00:00<?, ?it/s]


[INFO] user_id 01fc52c4bc9ffc8c: 3 categorías.


  1%|▏         | 1/73 [00:03<03:49,  3.19s/it]


[INFO] user_id 04d09936041bc786: 5 categorías.


...


[INFO] user_id ff1015b67c1a3b4a: 3 categorías.


 99%|█████████▊| 72/73 [04:30<00:03,  3.08s/it]


[INFO] user_id ff252803642ed916: 5 categorías.


100%|██████████| 73/73 [04:33<00:00,  3.75s/it]


✓ Clasificación completada. Total resultados: 73


In [ ]:
# CONSOLIDACIÓN FINAL 
def parse_json_field(v):
    try:
        return json.loads(v) if isinstance(v, str) else (v if isinstance(v, list) else [])
    except Exception:
        return []

df_ckpt_final = pd.read_csv(RUTA_CHECKPOINT, dtype={"user_id": str})
df_ckpt_final["categorias_list"] = df_ckpt_final["categorias"].apply(parse_json_field)
df_ckpt_final["n"] = df_ckpt_final["categorias_list"].apply(len)

# Deduplicamos quedándonos con la entrada con más categorías por usuario
df_ckpt_final = (
    df_ckpt_final.sort_values("n")
    .drop_duplicates(subset="user_id", keep="last")
    .reset_index(drop=True)
)

registros = []
for _, row in df_ckpt_final.iterrows():
    rec = {
        "user_id":       row["user_id"],
        "n_categorias":  int(row["n_categorias"]),
        "sin_categoria": bool(row["sin_categoria"]),
    }
    cats = row["categorias_list"] or []
    for idx in range(10):
        if idx < len(cats):
            rec[f"categoria_{idx+1}"] = cats[idx]["nombre_oficial"]
            rec[f"cat_id_{idx+1}"]    = cats[idx]["id"]
            rec[f"evidencia_{idx+1}"] = cats[idx].get("evidencia", "")
        else:
            rec[f"categoria_{idx+1}"] = ""
            rec[f"cat_id_{idx+1}"]    = ""
            rec[f"evidencia_{idx+1}"] = ""
    registros.append(rec)

df_plano = pd.DataFrame(registros)

df_final = tweets_filtrados.merge(
    df_plano, on="user_id", how="left"
)

df_final.to_csv(RUTA_RESULTADO, index=False)
print(f"✓ Resultado guardado: {RUTA_RESULTADO}")
print(f"Shape: {df_final.shape}\n")
print("Distribución de n_categorias:")
print(df_final["n_categorias"].value_counts().sort_index())
print(f"\nUsuarios con sin_categoria=True:  {df_final['sin_categoria'].sum()}")
print(f"Usuarios con al menos 1 categoría: {(df_final['n_categorias'] > 0).sum()}")
df_final.head()

✓ Resultado guardado: /content/drive/MyDrive/TFM/8. Categorizacion/llm_clasificacion_psico_final_prompt2.csv
Shape: (73, 46)

Distribución de n_categorias:
n_categorias
1     4
2    10
3    20
4    17
5    16
6     5
7     1
Name: count, dtype: int64

Usuarios con sin_categoria=True:  0
Usuarios con al menos 1 categoría: 73


,user_id,userName,name,tweets_concat,n_tweets,first_tweet,last_tweet,profile_bio,location,followers,...,evidencia_7,categoria_8,cat_id_8,evidencia_8,categoria_9,cat_id_9,evidencia_9,categoria_10,cat_id_10,evidencia_10
0,01fc52c4bc9ffc8c,USR_69f55ce3,[NOMBRE],[TWEET 1] Nacemos solos &amp; morimos en el al...,74,2025-07-22,2026-04-16,NaN,"<LOCATION>, <LOCATION>",788.0,...,,,,,,,,,,
1,04d09936041bc786,USR_22126242,[NOMBRE],[TWEET 1] RT [USUARIO]: Si ante la desgracia d...,100,2023-12-08,2026-04-16,"Psicóloga. Especializada en trauma, trastorn...",NaN,9186.0,...,,,,,,,,,,
2,05306b7fa750a5d8,USR_e1786264,[NOMBRE],[TWEET 1] Si no dice Israel en el título no es...,100,2025-07-28,2026-04-20,"Orgullosamente judio. Si sos nazi, zurdo o k n...","<LOCATION>, <LOCATION>",8515.0,...,,,,,,,,,,
3,0532b08c30376fde,USR_7ff3b9c0,[NOMBRE],[TWEET 1] llevo todo el día en la cama\n\n[TWE...,99,2024-01-21,2024-12-18,ig;:? ñ,"<LOCATION>, <LOCATION>",26.0,...,,,,,,,,,,
4,0588fd52a2609179,USR_8794a5e3,[NOMBRE],[TWEET 1] RT [USUARIO]: Optimus Christ [URL]\n...,98,2026-04-07,2026-04-20,Androide power!!! Obsesionado de la tecnología...,"<LOCATION>, <LOCATION>",709.0,...,,,,,,,,,,


In [ ]:
# Muestra para revisión manual
muestra = df_final.sample(min(5, len(df_final)), random_state=42)

for _, row in muestra.iterrows():
    print("="*80)
    print(f"user_id: {row['user_id']} | n_categorias: {row['n_categorias']} | sin_categoria: {row['sin_categoria']}")
    for i in range(1, 11):
        cat = row.get(f"categoria_{i}", "")
        if cat:
            ev  = row.get(f"evidencia_{i}", "")
            cid = row.get(f"cat_id_{i}", "")
            print(f"  [{cid}] {str(cat).strip()}  →  {str(ev)[:120]}")
    print()

user_id: 0588fd52a2609179 | n_categorias: 4 | sin_categoria: False
  [175] Anime y comics  →  Menciones recurrentes a series anime y manga
  [16] Objetos de colección y antiguedades  →  Interés en figuras, coleccionismo y cosplay
  [7] Negocios y economia  →  Crítica política y actualidad española
  [172] Entretenimiento y cultura  →  Debates sobre piratería y acceso a contenido

user_id: d631f1cf8e25eb1f | n_categorias: 5 | sin_categoria: False
  [176] Deportes  →  Seguidor recurrente y apasionado del Atlético Madrid
  [172] Entretenimiento y cultura  →  Comenta discos, canciones y artistas frecuentemente
  [17] Telenovelas y series de TV  →  Menciona ver series como The Boys, Euphoria
  [22] Lectura  →  Menciona Los Miserables y hábitos de lectura
  [32] Salud y bienestar  →  Menciona terapia, psicólogo y salud mental

user_id: 2527216c16349123 | n_categorias: 5 | sin_categoria: False
  [32] Salud y bienestar  →  Dolor crónico, fibromialgia, medicación y ansiedad.
  [179] Sociedad  →